# Étape 2 — Agent DDPG pour la gestion de stock

Ce notebook entraîne l'agent DDPG (`agent/ddpg.py`) sur l'environnement d'inventaire
(`env/inventory_env.py`), construit à partir du dataset Kaggle *Retail Store Inventory
Forecasting Dataset*.

**Rappel du principe :**
- État (9 dimensions) : niveau de stock, demande prévue, prix, prix concurrent, remise,
  indicateur promo/jour férié, saisonnalité (encodée en sin/cos), proportion de jours
  restants dans l'épisode.
- Action (1 dimension continue) : quantité à commander, bornée dans `[0, MAX_ORDER]`.
- Récompense : profit journalier (revenu des ventes − coût de commande − coût de
  stockage − coût de rupture de stock).
- Un épisode = la série temporelle complète (731 jours) d'un couple (magasin, produit)
  tiré aléatoirement à chaque `reset()`.

In [ ]:
import sys
sys.path.append("..")  

import numpy as np
import matplotlib.pyplot as plt

from agent.ddpg import DDPGAgent
from env.inventory_env import InventoryEnv
from utils import plot_training_curve

DATA_PATH = "../data/retail_store_inventory.csv"
SEED = 42
np.random.seed(SEED)

## 1. Initialisation de l'environnement et de l'agent

In [ ]:
env = InventoryEnv(DATA_PATH)  

agent = DDPGAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    action_low=env.action_low,
    action_high=env.action_high,
    actor_lr=1e-4,
    critic_lr=1e-3,
    gamma=0.99,
    tau=0.005,
)

print("state_dim :", env.state_dim)
print("action bounds :", env.action_low, "->", env.action_high)

## 2. Boucle d'entraînement

In [ ]:
N_EPISODES = 30
WARMUP_STEPS = 500
BATCH_SIZE = 128

total_steps = 0
episode_rewards = []

for episode in range(1, N_EPISODES + 1):
    state = env.reset(seed=SEED + episode)
    agent.noise.reset()
    ep_reward = 0.0
    done = False

    while not done:
        if total_steps < WARMUP_STEPS:
            action = np.random.uniform(env.action_low, env.action_high, size=(env.action_dim,))
        else:
            action = agent.select_action(state, explore=True)

        next_state, reward, done, info = env.step(action)
        agent.store_transition(state, action, reward, next_state, float(done))

        if total_steps >= WARMUP_STEPS:
            agent.update(batch_size=BATCH_SIZE)

        state = next_state
        ep_reward += reward
        total_steps += 1

    episode_rewards.append(ep_reward)
    if episode % 5 == 0 or episode == 1:
        print(f"Épisode {episode:3d}/{N_EPISODES} | récompense = {ep_reward:8.2f} | "
              f"moyenne (5 derniers) = {np.mean(episode_rewards[-5:]):8.2f}")

## 3. Courbe d'apprentissage

In [ ]:
plot_training_curve(episode_rewards, save_path="../results/plots/training_curve_notebook.png", window=5)

plt.figure(figsize=(8, 4))
plt.plot(episode_rewards)
plt.xlabel("Épisode")
plt.ylabel("Récompense cumulée")
plt.title("Récompense par épisode (agent DDPG)")
plt.grid(alpha=0.3)
plt.show()

## 4. Sauvegarde du modèle entraîné

In [ ]:
import os
os.makedirs("../results/models", exist_ok=True)
agent.save("../results/models/ddpg_notebook")
print("Modèle sauvegardé dans results/models/")

## 5. Inspection d'un épisode (politique apprise vs. demande réelle)

In [ ]:
eval_env = InventoryEnv(DATA_PATH, store_id="S001", product_id="P0001")
state = eval_env.reset(seed=0)

orders, demands, inventories = [], [], []
done = False
while not done:
    action = agent.select_action(state, explore=False)
    state, reward, done, info = eval_env.step(action)
    orders.append(info["order"])
    demands.append(info["demand"])
    inventories.append(info["inventory"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(demands, label="Demande réelle (Units Sold)", alpha=0.7)
ax.plot(orders, label="Quantité commandée par l'agent", alpha=0.7)
ax.plot(inventories, label="Niveau de stock", alpha=0.7)
ax.set_xlabel("Jour")
ax.legend()
ax.set_title("S001 / P0001 — politique apprise (sans exploration)")
plt.tight_layout()
plt.show()